In [3]:
from pathlib import Path
import pandas as pd
import json
import shutil
from google.colab import drive
drive.mount("/content/drive/")

PROJECT_DIR = Path("/content/drive/MyDrive/Underwater-Image-Data-set-main")
V1_DIR = PROJECT_DIR / "Dataset_V1"

FINAL_DIR = V1_DIR / "Final_Method"
FINAL_DIR.mkdir(parents=True, exist_ok=True)

TUNING_DIR = V1_DIR / "Classical_Tuning"
MODEL_DIR = V1_DIR / "Learning_Based_Model"
BASELINE_DIR = V1_DIR / "Classical_Baselines"
SPECIAL_DIR = V1_DIR / "Special_Analysis"

SPLIT_FILE = V1_DIR / "Dataset_V1_splits.csv"

print("FINAL METHOD SELECTION AND FREEZE")


df = pd.read_csv(SPLIT_FILE)

train_count = len(df[df["split"] == "train"])
val_count = len(df[df["split"] == "validation"])
test_count = len(df[df["split"] == "test"])

print("\nDataset split:")
print("Train:", train_count)
print("Validation:", val_count)
print("Test:", test_count)

best_config_file = (
    TUNING_DIR /
    "best_validated_method.json"
)

if not best_config_file.exists():
    raise FileNotFoundError(
        "best_validated_method.json was not found. Run parameter tuning first."
    )

with open(best_config_file, "r") as f:
    best_config = json.load(f)

classical_method = best_config["selected_method"]
classical_parameters = best_config["best_parameters"]

print("\nValidated classical method:", classical_method)
print("Validated parameters:", classical_parameters)

tuning_file = (
    TUNING_DIR /
    "parameter_tuning_results.csv"
)

tuning = pd.read_csv(tuning_file)

selected_rows = tuning[
    tuning["selected"] == True
].copy()

if len(selected_rows) == 0:
    selected_rows = (
        tuning
        .sort_values("Average_Rank")
        .head(1)
    )

selected = selected_rows.iloc[0]

validation_metrics = {
    "PSNR": float(selected["PSNR"]),
    "SSIM": float(selected["SSIM"]),
    "Edge Preservation": float(
        selected["Edge Preservation"]
    )
}

print("\nValidation evidence:")
print(validation_metrics)

special_summary_file = (
    SPECIAL_DIR /
    "classical_vs_learning_summary.csv"
)

learning_available = False
learning_metrics = {}

if special_summary_file.exists():

    special_summary = pd.read_csv(
        special_summary_file
    )

    learning_rows = special_summary[
        special_summary["condition"]
        == "Learning-based"
    ]

    if len(learning_rows) > 0:

        learning_available = True

        learning = learning_rows.iloc[0]

        for metric in [
            "PSNR",
            "SSIM",
            "Edge Preservation",
            "Edge F1",
            "Processing Time"
        ]:

            if metric in learning.index:
                learning_metrics[metric] = float(
                    learning[metric]
                )

print("\nLearning-based evidence available:", learning_available)

final_method = classical_method
final_parameters = classical_parameters
selection_basis = "Validation evidence, robustness checks, and methodological simplicity."

classical_robustness = {
    "brightness_artifact_check": True,
    "contrast_artifact_check": True,
    "edge_artifact_check": True,
    "parameter_search_documented": True,
    "test_used_for_selection": False
}

preprocessing_config = {
    "image_size": [224, 224],
    "image_range": "[0,1]",
    "train_augmentation": [
        "RandomHorizontalFlip(p=0.5)"
    ],
    "validation_augmentation": [],
    "test_augmentation": [],
    "normalization": {
        "mean": [
            0.485,
            0.456,
            0.406
        ],
        "std": [
            0.229,
            0.224,
            0.225
        ]
    }
}

evaluation_config = {
    "primary_metrics": [
        "PSNR",
        "SSIM",
        "Edge Preservation"
    ],
    "additional_metrics": [
        "Edge Precision",
        "Edge Recall",
        "Edge F1",
        "Processing Time"
    ],
    "test_set_used_for_selection": False,
    "final_evaluation_uses_frozen_test_set": True
}

split_config = {
    "split_file": str(SPLIT_FILE),
    "train_percentage": 70,
    "validation_percentage": 15,
    "test_percentage": 15,
    "split_unit": "sample_id",
    "triplets_kept_together": True,
    "random_state": 42,
    "frozen": True
}

selection_rationale = {
    "final_method": final_method,
    "final_parameters": final_parameters,
    "validation_metrics": validation_metrics,
    "selection_basis": selection_basis,
    "robustness_evidence": classical_robustness,
    "simplicity_reason": "The selected classical pipeline has fewer trainable components and lower methodological complexity than the supervised learning pipeline.",
    "learning_based_results_available": learning_available,
    "learning_based_metrics": learning_metrics,
    "test_set_used_for_selection": False,
    "final_test_evaluation": "To be performed only after the methodology is frozen."
}

with open(
    FINAL_DIR /
    "final_method_selection_rationale.json",
    "w"
) as f:
    json.dump(
        selection_rationale,
        f,
        indent=4
    )

with open(
    FINAL_DIR /
    "frozen_preprocessing_config.json",
    "w"
) as f:
    json.dump(
        preprocessing_config,
        f,
        indent=4
    )

with open(
    FINAL_DIR /
    "frozen_split_config.json",
    "w"
) as f:
    json.dump(
        split_config,
        f,
        indent=4
    )

with open(
    FINAL_DIR /
    "frozen_evaluation_config.json",
    "w"
) as f:
    json.dump(
        evaluation_config,
        f,
        indent=4
    )

final_config = {
    "final_method": final_method,
    "parameters": final_parameters,
    "preprocessing": preprocessing_config,
    "split": split_config,
    "evaluation": evaluation_config
}

with open(
    FINAL_DIR /
    "FINAL_PIPELINE_CONFIG.json",
    "w"
) as f:
    json.dump(
        final_config,
        f,
        indent=4
    )

shutil.copy2(
    SPLIT_FILE,
    FINAL_DIR /
    "Dataset_V1_splits_FROZEN.csv"
)

shutil.copy2(
    best_config_file,
    FINAL_DIR /
    "best_validated_method.json"
)

if tuning_file.exists():

    shutil.copy2(
        tuning_file,
        FINAL_DIR /
        "parameter_tuning_results.csv"
    )

model_file = (
    MODEL_DIR /
    "best_unet_model.pth"
)

if model_file.exists():

    shutil.copy2(
        model_file,
        FINAL_DIR /
        "best_unet_model.pth"
    )

training_config = (
    MODEL_DIR /
    "training_config.json"
)

if training_config.exists():

    shutil.copy2(
        training_config,
        FINAL_DIR /
        "training_config.json"
    )

training_history = (
    MODEL_DIR /
    "training_history.csv"
)

if training_history.exists():

    shutil.copy2(
        training_history,
        FINAL_DIR /
        "training_history.csv"
    )

final_outputs_dir = (
    FINAL_DIR /
    "Final_Outputs"
)

final_outputs_dir.mkdir(
    parents=True,
    exist_ok=True
)

candidate_output_dirs = [
    TUNING_DIR / "best_validated_outputs",
    MODEL_DIR / "validation_outputs"
]

for source_dir in candidate_output_dirs:

    if source_dir.exists():

        destination = (
            final_outputs_dir /
            source_dir.name
        )

        if destination.exists():
            shutil.rmtree(destination)

        shutil.copytree(
            source_dir,
            destination
        )

freeze_record = {
    "status": "FROZEN",
    "final_method": final_method,
    "final_parameters": final_parameters,
    "train_samples": train_count,
    "validation_samples": val_count,
    "test_samples": test_count,
    "split_frozen": True,
    "preprocessing_frozen": True,
    "evaluation_methodology_frozen": True,
    "test_used_for_method_selection": False,
    "selection_rationale_saved": True
}

with open(
    FINAL_DIR /
    "FREEZE_RECORD.json",
    "w"
) as f:
    json.dump(
        freeze_record,
        f,
        indent=4
    )
print("FINAL METHOD")

print("Method:", final_method)
print("Parameters:", final_parameters)

print("\nValidation evidence:")
for key, value in validation_metrics.items():
    print(
        key + ":",
        round(value, 4)
    )

print("\nFrozen:")
print("Preprocessing: YES")
print("Train/validation/test split: YES")
print("Evaluation methodology: YES")
print("Test used for selection: NO")

print("\nFinal artifacts saved to:")
print(FINAL_DIR)

print("\nFiles:")
for file in sorted(FINAL_DIR.iterdir()):
    print(" -", file.name)

print("\nFINAL PIPELINE IS FROZEN")

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
FINAL METHOD SELECTION AND FREEZE

Dataset split:
Train: 219
Validation: 47
Test: 48

Validated classical method: Gamma
Validated parameters: {'gamma': 0.8}

Validation evidence:
{'PSNR': 13.697720642425766, 'SSIM': 0.6623341905071979, 'Edge Preservation': 0.0058264320453484}

Learning-based evidence available: True
FINAL METHOD
Method: Gamma
Parameters: {'gamma': 0.8}

Validation evidence:
PSNR: 13.6977
SSIM: 0.6623
Edge Preservation: 0.0058

Frozen:
Preprocessing: YES
Train/validation/test split: YES
Evaluation methodology: YES
Test used for selection: NO

Final artifacts saved to:
/content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1/Final_Method

Files:
 - Dataset_V1_splits_FROZEN.csv
 - FINAL_PIPELINE_CONFIG.json
 - FREEZE_RECORD.json
 - Final_Outputs
 - best_unet_model.pth
 - best_validated_method.json
 - final_method_selection_rationale.js